### Lab 2

In [7]:
import pandas as pd
import numpy as np
import altair as alt


alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

### Chart 1 — Character Prominence & Episode Dialogue Distribution (Words)

The first dashboard pairs a horizontal bar chart of series-wide word totals with a boxplot of per-episode distributions, and the two charts share a click-based selection that links them. We separated the underlying data sources deliberately: the bar chart reads from pre-aggregated summary data while the boxplot reads from raw episode-level rows, which required us to set `fields=['name']` explicitly in the selection parameter — without it, Altair cannot infer the linking field across two distinct CSV sources. We called `alt.data_transformers.disable_max_rows()` because the episode dataset exceeds 94,000 rows, which Altair blocks by default. Before rendering, we filtered out three entries — "Entire Town," "ABBA," and "Robert Pinsky" — which are meta-labels and placeholders rather than real characters, and whose inflated word counts (one placeholder carried 571,000 words) corrupted the top-10 ranking entirely. We also applied a per-episode threshold of 2,500 words to remove outlier rows that were compressing the boxplot scale to the point of illegibility. For color, selected characters appear in blue (#1f77b4), unselected in grey (#d3d3d3), and boxplots consistently use green (#2ca02c) as a visual signal that they encode a different statistical layer. Row height is set via `alt.Step(40)` for adequately sized click targets, and the boxplot Y-axis carries `axis=None` to suppress repeated character names already visible on the left.

In [5]:
import pandas as pd
import altair as alt

df_agg = pd.read_csv('data/clean/agg_words_by_char.csv')
df_episode_raw = pd.read_csv('data/clean/agg_words_by_char_episode.csv')

exclude_list = ['Entire Town', 'ABBA', 'Robert Pinsky']
df_clean_agg = df_agg[~df_agg['name'].isin(exclude_list)]
df_top_10 = df_clean_agg.sort_values('total_words', ascending=False).head(10)
top_names = df_top_10['name'].unique().tolist()

df_episode_clean = df_episode_raw[
    (df_episode_raw['name'].isin(top_names)) & 
    (df_episode_raw['total_words'] < 2500)
].copy()

alt.data_transformers.disable_max_rows()
selection = alt.selection_point(fields=['name'], on='click')

base_bars = alt.Chart(df_top_10).mark_bar().encode(
    x=alt.X('total_words:Q', title='Total Spoken Words (Series Wide)'),
    y=alt.Y('name:N', sort=None, title='Character Selection'),
    color=alt.condition(selection, alt.value('#1f77b4'), alt.value('#d3d3d3'))
).add_params(selection)

text_labels = base_bars.mark_text(align='left', baseline='middle', dx=5).encode(text='total_words:Q')

chart1 = (base_bars + text_labels).properties(
    width=400,
    height=alt.Step(40)
)

chart2 = alt.Chart(df_episode_clean).mark_boxplot(outliers=True).encode(
    x=alt.X('total_words:Q', title='Words Spoken in a Single Episode'),
    y=alt.Y('name:N', sort=None, axis=None),
    color=alt.value('#2ca02c') 
).transform_filter(
    selection 
).properties(
    width=350,
    height=alt.Step(40)
)

dashboard = (chart1 | chart2).configure_view(
    stroke=None
).properties(
    title=alt.TitleParams(
        text="Character Prominence & Episode Dialogue Distribution",
        subtitle=["Clicking a character reveals their volatility: do they dominate every show, or stay in the background until a spotlight episode?", ""],
        anchor='start',
        fontSize=16
    )
)

dashboard.display()

alt.HConcatChart(...)

### Chart 2 — Dialogue Share Over Time

The second visualization asks whether a character's dialogue share has grown, shrunk, or stayed flat across the show's run, and the first design decision was to reject raw word counts as the encoding variable entirely. Homer speaks so much more than any other character that plotting absolute totals produces one dominant line and nine others that are effectively flat — the comparison becomes meaningless. Instead, we normalized each character's words per season against the total words spoken by all top-ten characters in that season, yielding a dialogue share percentage that allows fair cross-season comparison regardless of how much the show's overall volume varied. We also considered a slopegraph and a "shadow lines" background approach before rejecting both: a slopegraph would have shown only endpoint seasons and obscured the mid-run fluctuations we wanted to make visible, while rendering all ten characters simultaneously as background context would have produced an unreadable spaghetti plot across 26 seasons. The final layout uses a vertical bar selector on the left, where clicking a bar adds that character's trend line to the right panel and clicking again removes it. We implemented `on='click[!event.shiftKey]'` so that users can build up a multi-character comparison without holding Shift. Selected bars are solid dark blue (#1f77b4) and deselected bars dim to pale blue (#b9cde5). Tooltips expose character name, season, and share percentage for precise reading.

In [6]:
import pandas as pd
import altair as alt

df_ep = pd.read_csv('data/clean/agg_words_by_char_episode.csv')

exclude = ['Entire Town', 'ABBA', 'Robert Pinsky']
df_clean = df_ep[(~df_ep['name'].isin(exclude)) & (df_ep['total_words'] < 2500)].copy()
top_10 = df_clean.groupby('name')['total_words'].sum().nlargest(10).index.tolist()
df_clean = df_clean[df_clean['name'].isin(top_10)]

season_totals = df_clean.groupby('season')['total_words'].transform('sum')
df_clean['share_pct'] = (df_clean['total_words'] / season_totals) * 100
df_trends = df_clean.groupby(['name', 'season'])['share_pct'].sum().reset_index()

df_selector = df_trends.groupby('name')['share_pct'].mean().reset_index()

char_select = alt.selection_point(
    fields=['name'], 
    on='click[!event.shiftKey]', 
    toggle='!event.shiftKey', 
    empty='all'
)

left_bars = alt.Chart(df_selector).mark_bar().encode(
    x=alt.X('share_pct:Q', title='Avg Dialogue Share (%)'),
    y=alt.Y('name:N', sort='-x', title=None),
    color=alt.condition(char_select, alt.value('#1f77b4'), alt.value('#b9cde5'))
).add_params(char_select).properties(
    width=180, 
    height=400, 
    title="Click Bars to Add/Remove"
)

right_trends = alt.Chart(df_trends).mark_line(
    strokeWidth=3, 
    point=True
).encode(
    x=alt.X('season:O', title='Season Timeline', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('share_pct:Q', title='Dialogue Share Percentage (%)'),
    color=alt.Color('name:N', title='Character', scale=alt.Scale(scheme='category10')),
    tooltip=['name', 'season', 'share_pct']
).transform_filter(
    char_select  
).properties(
    width=550, 
    height=400, 
    title="Historical Dialogue Share Tracking"
)

dashboard = (left_bars | right_trends).configure_view(
    stroke=None
)

dashboard.display()



alt.HConcatChart(...)

### Chart 3 — Character Prominence by Sentences Spoken

The third dashboard mirrors the structure of the first but shifts the metric from total words to total sentences, making it possible to ask a different question: which characters speak most frequently, as opposed to at greatest length. Because the two charts share the same visual grammar — bar chart on the left, boxplot on the right, linked by a click selection — a reader who has already used the first dashboard can immediately understand how to operate this one, and the design cost of learning a new interaction pattern is zero. We preserved the identical color scheme: selected bars in blue (#1f77b4), unselected in grey (#d3d3d3), boxplots in green (#2ca02c), so the same color always encodes the same role across both dashboards and the green-for-distribution convention carries its meaning without needing to be re-established. The same `fields=['name']` selection binding applies here for the same reason as in Chart 1 — two separate data sources require an explicit linking field. We applied `alt.Step(40)` row heights and `axis=None` on the boxplot Y-axis for the same reasons of click-target size and label redundancy. Titles and subtitles were updated to reflect the sentence-based metric so that neither dashboard is ambiguous about what it is measuring.

In [1]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

df_agg = pd.read_csv('data/clean/agg_words_by_char.csv')
df_ep  = pd.read_csv('data/clean/agg_words_by_char_episode.csv')

exclude = ['Entire Town', 'ABBA', 'Robert Pinsky']
df_top10 = (
    df_agg[~df_agg['name'].isin(exclude)]
    .sort_values('total_sentences', ascending=False)
    .head(10)
)
top_names = df_top10['name'].unique().tolist()
df_ep_clean = df_ep[df_ep['name'].isin(top_names)].copy()

selection = alt.selection_point(fields=['name'], on='click')

# --- CHART 1: total sentences bar ---
bars = alt.Chart(df_top10).mark_bar().encode(
    x=alt.X('total_sentences:Q', title='Total Sentences Spoken (Series Wide)'),
    y=alt.Y('name:N', sort=None, title='Character'),
    color=alt.condition(selection, alt.value('#1f77b4'), alt.value('#d3d3d3'))
).add_params(selection)

labels = bars.mark_text(align='left', baseline='middle', dx=5).encode(
    text='total_sentences:Q'
)

chart1 = (bars + labels).properties(width=400, height=alt.Step(40))

chart2 = alt.Chart(df_ep_clean).mark_boxplot(outliers=True).encode(
    x=alt.X('total_sentences:Q', title='Sentences Spoken in a Single Episode'),
    y=alt.Y('name:N', sort=None, axis=None),
    color=alt.value('#2ca02c')
).transform_filter(selection).properties(width=350, height=alt.Step(40))

(chart1 | chart2).configure_view(stroke=None).properties(
    title=alt.TitleParams(
        text='Character Prominence by Sentences Spoken',
        subtitle=['Left: total utterances series-wide — click to reveal episode distribution on the right', ''],
        anchor='start',
        fontSize=16
    )
).display()


alt.HConcatChart(...)